# DINOv2 Feature Visualization

## Setup

In [ ]:
import subprocess, os, sys

UV = '/root/.local/bin/uv'

subprocess.run([
    UV, 'pip', 'install', '--system',
    'torch', 'torchvision',
    'opencv-python-headless',
    'matplotlib',
    'numpy',
    'Pillow',
    'tqdm',
    'scikit-learn',
    'requests',
], check=True)

In [ ]:
import subprocess, os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from tqdm import tqdm
from sklearn.decomposition import PCA
from torchvision import transforms

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 1. Extract Frames

In [ ]:
import requests

VIDEO_URL  = 'https://www.dropbox.com/scl/fi/b3bdak9jdrbz1j063gaam/scratch_3.mp4?rlkey=qvrfed1dpwqz4g0rprenebips&dl=1'
VIDEO_PATH = '/workspace/scratch_3.mp4'
FRAMES_DIR = '/workspace/frames'
OUTPUT_DIR = '/workspace/dinov2_output'
os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(VIDEO_PATH):
    print('Downloading video...')
    r = requests.get(VIDEO_URL, stream=True)
    r.raise_for_status()
    with open(VIDEO_PATH, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print('Done.')
else:
    print('Already downloaded.')

FRAME_STRIDE = 5
IMG_SIZE = 448

def center_crop_resize(frame_bgr, size=IMG_SIZE):
    img = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    w, h = img.size
    s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    return img.resize((size, size), Image.LANCZOS)

cap = cv2.VideoCapture(VIDEO_PATH)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps   = cap.get(cv2.CAP_PROP_FPS)
start_frame = total // 4
print(f'{total} frames @ {fps:.1f} fps, extracting from frame {start_frame}')

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
frame_paths = []
idx = saved = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if idx % FRAME_STRIDE == 0:
        img = center_crop_resize(frame)
        path = f'{FRAMES_DIR}/{saved:04d}.png'
        img.save(path)
        frame_paths.append(path)
        saved += 1
    idx += 1
cap.release()
print(f'Saved {saved} frames')

## 2. Load DINOv2

In [ ]:
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
model = model.to(DEVICE).eval()

PATCH_SIZE = 14
N_PATCHES  = IMG_SIZE // PATCH_SIZE  # 32
print(f'Patch grid: {N_PATCHES}x{N_PATCHES}')

preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

def get_features(img_path):
    img = Image.open(img_path).convert('RGB')
    x = preprocess(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = model.forward_features(x)
    return out['x_norm_patchtokens'].squeeze(0)  # (N, D)

## 3. Feature Similarity (query point -> heatmap)

In [ ]:
QUERY_X = 324
QUERY_Y = 243
QUERY_FRAME_IDX = 0

qr = QUERY_Y // PATCH_SIZE
qc = QUERY_X // PATCH_SIZE
q_idx = qr * N_PATCHES + qc
print(f'Query patch ({qr}, {qc}), flat index {q_idx}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(Image.open(frame_paths[QUERY_FRAME_IDX]))
ax.add_patch(plt.Rectangle((qc*PATCH_SIZE, qr*PATCH_SIZE), PATCH_SIZE, PATCH_SIZE,
                             linewidth=2, edgecolor='red', facecolor='red', alpha=0.4))
ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/query_location.png', dpi=150)
plt.show()

In [ ]:
q_feats = get_features(frame_paths[QUERY_FRAME_IDX])
q_vec   = F.normalize(q_feats[q_idx].unsqueeze(0), dim=-1)  # (1, D)

SIM_DIR = f'{OUTPUT_DIR}/similarity'
os.makedirs(SIM_DIR, exist_ok=True)

VIZ = frame_paths[::2][:40]

for fi, fpath in enumerate(tqdm(VIZ)):
    feats    = F.normalize(get_features(fpath), dim=-1)
    sim      = (feats @ q_vec.T).squeeze(-1).cpu().numpy()
    sim_map  = sim.reshape(N_PATCHES, N_PATCHES)
    sim_img  = np.kron(sim_map, np.ones((PATCH_SIZE, PATCH_SIZE)))
    sim_img  = (sim_img - sim_img.min()) / (sim_img.max() - sim_img.min() + 1e-8)

    frame_np = np.array(Image.open(fpath))
    heat     = (cm.inferno(sim_img)[:, :, :3] * 255).astype(np.uint8)
    overlay  = (0.45 * frame_np + 0.55 * heat).astype(np.uint8)

    if fi == 0:
        cv2.rectangle(overlay,
                      (qc*PATCH_SIZE, qr*PATCH_SIZE),
                      ((qc+1)*PATCH_SIZE, (qr+1)*PATCH_SIZE),
                      (255, 0, 0), 2)

    Image.fromarray(overlay).save(f'{SIM_DIR}/{fi:04d}.png')

subprocess.run([
    'ffmpeg', '-y', '-framerate', '10',
    '-i', f'{SIM_DIR}/%04d.png',
    '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
    f'{OUTPUT_DIR}/similarity.mp4'
], check=True)
print('Done: similarity.mp4')

## 4. PCA Visualization

In [ ]:
all_feats = np.concatenate([
    get_features(fpath).cpu().numpy()
    for fpath in tqdm(frame_paths[::3][:60], desc='Collecting features')
], axis=0)
print(f'Feature matrix: {all_feats.shape}')

pca = PCA(n_components=3)
pca.fit(all_feats)
print(f'Explained variance: {pca.explained_variance_ratio_}')

In [ ]:
PCA_DIR = f'{OUTPUT_DIR}/pca'
os.makedirs(PCA_DIR, exist_ok=True)

for fi, fpath in enumerate(tqdm(frame_paths[::2][:40], desc='PCA frames')):
    feats = get_features(fpath).cpu().numpy()
    proj  = pca.transform(feats)
    proj  = (proj - proj.min(0)) / (proj.max(0) - proj.min(0) + 1e-8)
    pca_map = np.kron(proj.reshape(N_PATCHES, N_PATCHES, 3),
                      np.ones((PATCH_SIZE, PATCH_SIZE, 1)))
    pca_img  = (pca_map * 255).astype(np.uint8)
    frame_np = np.array(Image.open(fpath))
    Image.fromarray(np.concatenate([frame_np, pca_img], axis=1)).save(f'{PCA_DIR}/{fi:04d}.png')

subprocess.run([
    'ffmpeg', '-y', '-framerate', '10',
    '-i', f'{PCA_DIR}/%04d.png',
    '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
    f'{OUTPUT_DIR}/pca.mp4'
], check=True)
print('Done: pca.mp4')

## 5. Preview

In [ ]:
idxs = [0, 5, 10, 15]
fig, axes = plt.subplots(2, len(idxs), figsize=(4*len(idxs), 8))

for col, i in enumerate(idxs):
    sim_f = Image.open(f'{SIM_DIR}/{i:04d}.png')
    pca_f = Image.open(f'{PCA_DIR}/{i:04d}.png')
    w = pca_f.width // 2
    axes[0, col].imshow(sim_f); axes[0, col].axis('off'); axes[0, col].set_title(f'Sim {i}')
    axes[1, col].imshow(pca_f.crop((w, 0, pca_f.width, pca_f.height)))
    axes[1, col].axis('off'); axes[1, col].set_title(f'PCA {i}')

plt.suptitle('Top: similarity  |  Bottom: PCA features')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/preview.png', dpi=150)
plt.show()